# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moham882/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
%pip -q install duckdb

In [4]:
import duckdb

con = duckdb.connect()

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


## 1. Unit of analysis + time window

*One row represents one content item for one client on one report date. I will work with the March 2026 reporting window.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Feature: search and performance fields that are knowable before the decision moment.

Label / proxy: the future performance outcome used to identify declining content; it is computed from outcome information and must never be used as a feature.

Context: client_id, content_id, and report_date, used for grouping, joining, splitting, and reading the data.

Excluded: trend_pct and trend_direction, because they are used to derive the decline label and would cause label leakage if used as features.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {REL}
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""")

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

In [11]:
slice_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {REL}
""")

slice_check

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘

In [12]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) AS ga4_available_rows
FROM {REL}
""")

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            9841378 │            6822637 │
└────────────┴────────────────────┴────────────────────┘

In [13]:
schema_check = con.sql(f"""
SELECT *
FROM {REL}
LIMIT 1
""")

schema_check

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

### Five-feature frame

1. `gsc_impressions` — knowable at the decision moment because it records search impressions already observed for the content.
2. `gsc_clicks` — knowable at the decision moment because it records clicks already observed for the content.
3. `gsc_avg_position` — knowable at the decision moment because it records the observed search position for the content.
4. `ga4_sessions` — knowable at the decision moment because it records sessions already observed, when GA4 data is available.
5. `sessions_organic` — knowable at the decision moment because it records organic sessions already observed, when the corresponding GA4 data is available.

GA4-based features have partial availability in the March 2026 slice, so missing availability must be handled explicitly rather than treated as evidence of zero activity.

In [15]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    sessions_organic
FROM {REL}
WHERE gsc_data_available IS TRUE
""").df()

features.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_organic
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


In [16]:
leakage_demo = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    -- DELIBERATE LABEL LEAKAGE FOR THE TRAP
    CASE
        WHEN gsc_impressions < 10 THEN 1
        ELSE 0
    END AS leaked_label_feature

FROM {REL}
WHERE gsc_data_available IS TRUE
""").df()

leakage_demo.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,leaked_label_feature
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,1
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,1
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0


In [17]:
import pandas as pd

REL_MARCH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
REL_APRIL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')"

march = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions
FROM {REL_MARCH}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

april = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS april_impressions
FROM {REL_APRIL}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

label_data = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

label_data["is_declining"] = (
    label_data["april_impressions"] < 0.8 * label_data["march_impressions"]
).astype(int)

label_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,april_impressions,is_declining
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,1151.0,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,73.0,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,98.0,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,2275.0,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,6266.0,0


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = label_data[["is_declining"]]
y = label_data["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leak_model = DecisionTreeClassifier(random_state=42)
leak_model.fit(X_train, y_train)

pred = leak_model.predict(X_test)

leak_score = accuracy_score(y_test, pred)

print(f"LEAKY MODEL ACCURACY: {leak_score:.3f}")

LEAKY MODEL ACCURACY: 1.000


### Leakage trap and correction

I deliberately used `is_declining`, the future outcome label, as a model feature. The model achieved an artificially near-perfect score because it was given information that directly defines the target.

I removed `is_declining` from the feature set. The correct approach is to use only information available at the decision moment and keep the future outcome as the label.

**Leakage lesson:** A very high model score can be caused by target leakage rather than genuine predictive ability. Features derived from the future outcome or label must never be used as model inputs.

### Limitation

The March 2026 development slice has incomplete GA4 availability. Only 6,822,637 of 9,841,378 rows have GA4 available, so GA4-based features cannot be assumed to be available for every content item.

## 4. Data limits

This data can never fully tell us:

- **Unbalanced history:** Not every client has the same amount of historical data, so comparisons across clients may not be equally reliable.
- **GSC-only early rows:** Some earlier rows have GSC data but no usable GA4 data, so the dataset cannot fully describe user behavior for those rows.
- **Window overlaps:** Different reporting windows can overlap, so features and labels must be aligned carefully to avoid using outcome information from the prediction period.
- **Causation:** The data can show patterns and predictive relationships, but it cannot by itself prove that one factor caused another.

In [19]:
data_limits = """
## 4. Data limits

This data can never fully tell us:

- Unbalanced history: Not every client has the same amount of historical data, so comparisons across clients may not be equally reliable.
- GSC-only early rows: Some earlier rows have GSC data but no usable GA4 data, so the dataset cannot fully describe user behavior for those rows.
- Window overlaps: Different reporting windows can overlap, so features and labels must be aligned carefully to avoid using outcome information from the prediction period.
- Causation: The data can show patterns and predictive relationships, but it cannot by itself prove that one factor caused another.
"""

print(data_limits)


## 4. Data limits

This data can never fully tell us:

- Unbalanced history: Not every client has the same amount of historical data, so comparisons across clients may not be equally reliable.
- GSC-only early rows: Some earlier rows have GSC data but no usable GA4 data, so the dataset cannot fully describe user behavior for those rows.
- Window overlaps: Different reporting windows can overlap, so features and labels must be aligned carefully to avoid using outcome information from the prediction period.
- Causation: The data can show patterns and predictive relationships, but it cannot by itself prove that one factor caused another.



## Self-check

Before you submit, confirm each line honestly:

-✅Every section above is filled — markdown thinking AND the code that backs it

-✅The notebook runs top to bottom with no errors (Runtime → Run all)

-✅No client names, URLs, or private queries anywhere

-✅My claims use careful words: observed, measured, directional, decision-support

-✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.